# ⚡ Kilat: Parameter-Efficient Fine-Tuning (PEFT) with LoRA

```text
  ██╗  ██╗██╗██╗      █████╗ ████████╗
  ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
  █████╔╝ ██║██║     ███████║   ██║   
  ██╔═██╗ ██║██║     ██╔══██║   ██║   
  ██║  ██╗██║███████╗██║  ██║   ██║   
  ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
  ────────────────────────────────────────────────────────────
  ⚡ KILAT | Kernelized Lightweight Attention Transformer (v2.0)
  Architecture: Hybrid Sub-Quadratic Global Decay + Latent MLA
  ────────────────────────────────────────────────────────────
```

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Airukua/Kilat/blob/main/notebooks/kilat_peft_lora_example.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Airukua%2FKilat-181717?style=flat-square&logo=github)](https://github.com/Airukua/Kilat)
[![Hugging Face](https://img.shields.io/badge/🤗%20HuggingFace-AiRukua%2F150M--Kilat--SMALL--10B-FFD21E?style=flat-square)](https://huggingface.co/AiRukua/150M-Kilat-SMALL-10B)

Welcome to the **Kilat PEFT LoRA Fine-Tuning** guide!
Low-Rank Adaptation (**LoRA**) allows adapting a pre-trained Kilat foundation model to downstream domain tasks by training low-rank decomposition matrices while keeping the base model weights completely frozen.

In this notebook, you will:
1. Load the pre-trained `AiRukua/150M-Kilat-SMALL-10B` base model.
2. Target Kilat's Latent MLA attention projections (`q_proj`, `kv_b_proj`, `out_proj`).
3. Verify that trainable parameters represent **< 1%** of total weights.
4. Fine-tune on custom instruction data.
5. Merge the adapter weights back into the base model (`merge_and_unload()`) for zero-overhead inference.

## 1. Environment Setup & Dependencies

In [ ]:
!rm -rf Kilat
!git clone https://github.com/Airukua/Kilat.git
%cd Kilat
!pip install -q -e ".[integrations]"
!pip install -q peft accelerate transformers

from kilat.utils.banner import print_kilat_banner
print_kilat_banner()

## 2. Load Base Kilat Model
We load the pre-trained foundation model weights from Hugging Face Hub.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "AiRukua/150M-Kilat-SMALL-10B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

if torch.cuda.is_available():
    model = model.cuda()

print("✅ Base Model Loaded successfully!")

## 3. Configure Kilat LoRA Adapter
Kilat provides built-in helpers in `kilat.integrations.huggingface.peft`:
- `DEFAULT_KILAT_LORA_TARGET_MODULES`: Targets attention projection layers (`q_proj`, `kv_b_proj`, `out_proj`).
- `get_kilat_lora_config`: Prepares a validated `LoraConfig`.

In [ ]:
from peft import get_peft_model
from kilat.integrations.huggingface.peft import get_kilat_lora_config, DEFAULT_KILAT_LORA_TARGET_MODULES

print("Target LoRA Modules:", DEFAULT_KILAT_LORA_TARGET_MODULES)

# Create LoRA configuration
peft_config = get_kilat_lora_config(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none"
)

# Wrap base model with PEFT LoRA adapter
peft_model = get_peft_model(model, peft_config)

# Print trainable vs total parameter summary
peft_model.print_trainable_parameters()

## 4. Invariant Verification: Frozen Base Parameters
Kilat's Invariant 2 requires that frozen parameters must have `requires_grad=False` and receive no gradient corruption.

In [ ]:
trainable_names = []
frozen_names = []

for name, param in peft_model.named_parameters():
    if "lora_" in name:
        assert param.requires_grad, f"LoRA param {name} should be trainable!"
        trainable_names.append(name)
    else:
        assert not param.requires_grad, f"Base param {name} should be frozen!"
        frozen_names.append(name)

print(f"⚡ Invariant Verified:")
print(f"• Frozen Base Parameters:     {len(frozen_names)}")
print(f"• Trainable Adapter Matrices: {len(trainable_names)}")

## 5. Fine-Tuning Step Example
Run forward pass, compute cross-entropy loss, and backpropagate to adapter parameters.

In [ ]:
# Sample instruction prompt
prompt = "Instruction: Explain sub-quadratic attention in one sentence.\nResponse:"
inputs = tokenizer(prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

# Set model to training mode
peft_model.train()
outputs = peft_model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss
loss.backward()

print("⚡ Training Step Loss:", loss.item())
print("✅ Adapter Gradients computed successfully without modifying frozen base weights.")

## 6. Merge Adapter Weights for Zero-Overhead Inference (`merge_and_unload`)
Once fine-tuning is complete, merge LoRA matrices directly into the base weights so inference runs with zero latency overhead.

In [ ]:
# Merge LoRA weights back into the base model
merged_model = peft_model.merge_and_unload()
merged_model.eval()

# Generate text with merged model
with torch.no_grad():
    gen_tokens = merged_model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print("⚡ [GENERATED]:\n", tokenizer.decode(gen_tokens[0], skip_special_tokens=True))